# 04 — Đánh giá E01–E04

Đánh giá từng run trên một tập, rồi so sánh. Báo cáo hai cấp: **Macro F1 tổng thể 22 lớp** (chỉ số chính) và
**Macro F1 nhóm 6 lớp liên quan nhựa** (`trash_bag`, `trash_bottle`, `trash_snack_wrapper`, `trash_net`, `trash_rope`, `trash_tarp`).

- Keras (E01–E02, `best.keras`): `src/evaluate.evaluate_run`
- PyTorch (E03–E04, `best.pt`): `src/evaluate.evaluate_torch_checkpoint`
- Mỗi run được đánh giá trên đúng phiên bản dữ liệu của nó (`v1_crop128` hoặc `v2_crop224`).

**Mặc định chỉ dùng Validation.** Tập Test chạy **một lần** cho mô hình cuối, sau khi Yến xác nhận (đặt `SPLIT = "test"`).
E04 dùng cách chia mới nên Validation/Test của E04 khác E01–E03.

Kết quả: `results/metrics/<EXP_ID>_<split>.json`, `<EXP_ID>_<split>_predictions.csv`, `results/figures/...`.

> Notebook này **chỉ gọi hàm trong `src/`**: không `%%writefile`, không chép code nguồn vào cell, không token, không `git push`.

## 1. Cài đặt và dữ liệu

In [ ]:
# Cài đặt chung: Colab → mount Drive + clone repo; máy cá nhân → dùng repo đang mở.
import os, sys, subprocess, json, time
from pathlib import Path

BRANCH = "main"   # đổi thành "feat/s2-pipeline-end-to-end" nếu PR #21 chưa merge
REPO_URL = "https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/ocean-plastic-ml")   # thư mục chung của nhóm
    WORK = Path("/content")                                   # ổ đĩa Colab, đọc nhanh hơn Drive
    REPO = WORK / "repo"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = REPO                                              # data/, runs/ đã nằm trong .gitignore
    WORK = REPO / "data"
os.chdir(REPO); sys.path.insert(0, str(REPO))
for sub in ["data/raw", "data/processed", "runs"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print("Repo:", REPO, "| commit:",
      subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Dữ liệu đã xử lý (tạo bởi notebook 02). Colab: giải nén bản .zip trên Drive nếu chưa có.
def processed_dir(data_version: str) -> Path:
    p = WORK / "processed" / data_version
    z = DRIVE / "data/processed" / f"{data_version}.zip"
    if IN_COLAB and not p.exists() and z.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", str(z), "-d", str(p.parent)], check=True)
    assert (p / "split_manifest.csv").exists(), f"Chưa có {p} — chạy notebook 02 trước"
    return p

## 2. Chọn run cần đánh giá

In [ ]:
SPLIT = "val"            # "test" chỉ MỘT lần cho mô hình cuối, sau khi Yến xác nhận

RUNS = DRIVE / "runs"
RUN_DIRS = []   # ví dụ [RUNS / "E01_20261105-1513", RUNS / "E04_20261121-1000"]; để trống → run mới nhất của mỗi EXP_ID
if not RUN_DIRS:
    latest = {}
    for d in sorted(RUNS.glob("E*_*")):
        if (d / "best.keras").exists() or (d / "best.pt").exists():
            latest[d.name.split("_")[0]] = d
    RUN_DIRS = list(latest.values())
assert RUN_DIRS, f"Không có run nào có best.keras / best.pt trong {RUNS} — chạy notebook 03 trước"
print("\n".join(map(str, RUN_DIRS)))

## 3. Đánh giá

In [ ]:
from src.evaluate import evaluate_run, evaluate_torch_checkpoint, comparison_table

RESULTS = REPO / "results"
res = {}
for run in RUN_DIRS:
    rcfg = json.loads((run / "config.json").read_text(encoding="utf-8"))
    if (run / "best.pt").exists():                      # E03–E04 (PyTorch)
        r = evaluate_torch_checkpoint(run, "best.pt", processed_dir(rcfg["data"]["data_version"]), split=SPLIT,
                                      out_dir=RESULTS / "metrics")
    else:                                               # E01–E02 (Keras)
        r = evaluate_run(run, processed_dir(rcfg["data_version"]), split=SPLIT, results_dir=RESULTS)
    res[r["exp_id"]] = r
    g = r["plastic_related"]
    print(f"{r['exp_id']} | n={r['num_samples']} | Macro F1 22 lớp={r['overall']['macro_f1']:.4f}"
          f" | Macro F1 nhóm nhựa={g['macro_f1'] if g['macro_f1'] is None else round(g['macro_f1'], 4)}")

## 4. So sánh thí nghiệm

In [ ]:
from IPython.display import display
table = comparison_table(sorted((RESULTS / "metrics").glob(f"E*_{SPLIT}.json")))
table.to_csv(RESULTS / "metrics" / f"comparison_{SPLIT}.csv", index=False)
display(table)

## 5. Chỉ số theo lớp (kèm support — lớp support nhỏ không kết luận mạnh)

In [ ]:
import pandas as pd
for exp_id, r in res.items():
    per_class = pd.DataFrame(r["overall"]["per_class"]).T[["precision", "recall", "f1_score", "support"]]
    print(exp_id)
    display(per_class.sort_values("support", ascending=False).round(3))

## 6. Confusion matrix, learning curves, mẫu dự đoán sai

In [ ]:
from IPython.display import Image
for exp_id in res:
    for name in [f"{exp_id}_{SPLIT}_confusion_matrix.png", f"{exp_id}_learning_curves.png", f"{exp_id}_{SPLIT}_misclassified.png"]:
        p = RESULTS / "figures" / name
        if p.exists():
            print(name); display(Image(filename=str(p), width=900))

## Phụ lục — Smoke test `src/evaluate.py` (dữ liệu giả lập)

Kiểm tra công thức chỉ số trên mảng nhỏ có đáp án biết trước. **Đây là dữ liệu giả lập, không phải kết quả trên TrashCan.**

In [ ]:
from src.evaluate import smoke_test
assert smoke_test()